In [34]:
import websocket, json
import pandas as pd
import os
import pickle, codecs
import asyncio
import nest_asyncio
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn import svm

In [ ]:
df = pd.read_csv('Biscotti_italia.csv')
df.reset_index(inplace=True, drop=False)

In [3]:
#Encodes the dataframe obj into bytes and then into a string
pickled = codecs.encode(pickle.dumps(df), "base64").decode()

In [4]:
#Takes the string and runs an undo over the procedure which we have just done
unpickled = pickle.loads(codecs.decode(pickled.encode(), "base64"))

In [36]:
import seaborn as sns
df = sns.load_dataset('iris')

In [37]:
X = df.drop('species', axis=1).values
y = df['species'].values

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [ ]:
parameters = {'kernel':('linear', 'rbf'), 'C':[1, 10]}
svc = svm.SVC()
clf = GridSearchCV(svc, parameters)
clf.fit(X, y)

In [40]:
clf.best_params_

{'C': 1, 'kernel': 'linear'}

In [41]:
X_train_pickled = codecs.encode(pickle.dumps(X_train), "base64").decode()
X_test_pickled = codecs.encode(pickle.dumps(X_test), "base64").decode()
y_train_pickled = codecs.encode(pickle.dumps(y_train), "base64").decode()
y_test_pickled = codecs.encode(pickle.dumps(y_test), "base64").decode()

In [250]:
ws = websocket.WebSocket()
ws.connect('ws://localhost:8000/ws/computing/')

In [42]:
payload = {
    "metadata": {
        "worker_node": 'a',
        "inquiring_node": 'b',
        "start_time": 'c',
        "max_cpu_allowed": .67,
        "timeout_boundary": 10,
        "task_type": 'a',
    },
    "data": {
        "X": X_train_pickled,
        "y": y_train_pickled,
    },
    "instructions": {
        "code" : 'i',
        "algorithm_name":'random_forest',
        "param_grid":{'n_estimators': [5, 10, 50, 100], 'criterion': ['giny', 'entropy']},
    },
}

In [255]:
!pip3 install websockets

     |████████████████████████████████| 94 kB 2.9 MB/s eta 0:00:011


In [261]:
!pip3 install nest-asyncio

In [43]:
nest_asyncio.apply()

In [50]:
#!/usr/bin/env python

async def send_data():
    async with websockets.connect('ws://192.168.1.227:8000/ws/computing/') as websocket:
        pp = json.dumps(payload).encode('utf-8')
        await websocket.send(pp)
        response = await websocket.recv()
        print(response)

r = asyncio.get_event_loop().run_until_complete(send_data())

{"best_params": {"criterion": "entropy", "n_estimators": 5}, "best_score": 0.9428571428571428}
